# 03 — Patient-level endpoint extraction

Builds **one auditable long table** of per-patient CellPhoneDB endpoints from the 24
per-patient runs produced by `02_run_cellphonedb.ipynb`.

**No pooled significance information is used anywhere in this notebook.** Nothing is read
from `cpdb_runs/pooled*` or `cpdb_runs/all`. Endpoint selection must not be contaminated by
the exploratory pooled comparison.

## Endpoint definition

An endpoint key is `(id_cp_interaction, sender, receiver)`. Cell-pair columns are
`"<sender>|<receiver>"` and **direction is preserved**: `A|B` (A sending to B) is a different
endpoint from `B|A`.

## Missingness semantics — the whole point of this notebook

| status | meaning |
|---|---|
| `evaluated` | sender and receiver both passed that patient's 10-cell threshold **and** CellPhoneDB computed a finite score |
| `structural_missing` | sender or receiver did not pass that patient's threshold, so the endpoint is undefined for that patient |
| `unexpected_missing` | both cell types retained, but the row or score is absent |
| `invalid` | duplicate endpoint key, non-numeric or non-finite score |

**Structural missingness is never converted to zero.** A score CellPhoneDB actually computed
as `0.0` stays `0.0` and is `evaluated`; a patient who simply lacks the cell type stays
missing. The legacy `cpdb_persample_consistency.py` collapses these two meanings into one
matrix — it writes a hard `0.0` for "present but never significant"
(`sample_score_sig[iid] = 0.0`) while keeping `NaN` for absent files — so a structural
absence and a genuine zero become indistinguishable downstream. That is exactly the bug this
notebook exists to avoid. It also hardcodes its Good/Poor patient lists; here every patient
is derived from the input tree.

## Expected `unexpected_missing`, and why

CellPhoneDB drops an interaction row entirely from a sample's output when neither participant
clears the expression filter in that sample, so per-patient `interaction_scores.csv` row
counts differ (527-1,189 in the legacy runs). Relative to the union of interactions across
patients, those absent rows are formally `unexpected_missing`. Gate 3 requires every one of
them to be explained by exactly that mechanism — the interaction is absent for **all** of that
patient's available cell pairs, never for some — and requires zero unexplained instances.

## Outputs

`patient_endpoint_scores_long.parquet` (authoritative), `patient_endpoint_availability.parquet`,
`endpoint_metadata.parquet`, `endpoint_qc.csv`, `patient_secondary_endpoint_scores.parquet`,
and a wide matrix only if it fits in memory.

Outcome labels are joined **only after** endpoint construction and QC are complete.

In [ ]:
import os
# Project data root. Override with the ACC_DATA_ROOT environment variable.
ACC_DATA_ROOT = os.environ.get("ACC_DATA_ROOT", os.path.expanduser("~/acc_data"))


## 1. Configuration (single config cell)

In [ ]:
# ============================================================
# CONFIG CELL — the only place values are declared.
# ============================================================
import gc
import inspect
import json
import os
import sys
import warnings
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

warnings.filterwarnings("ignore")

CODE_ROOT = Path(
    f"{ACC_DATA_ROOT}/scripts/named_scripts/cellphoneDB_scripts/clinical_outcome_v2"
)
CONFIG_PATH = CODE_ROOT / "config.yaml"
assert CODE_ROOT.is_dir(), CODE_ROOT

if str(CODE_ROOT) not in sys.path:
    sys.path.insert(0, str(CODE_ROOT))

import utils  # noqa: E402
from utils import (  # noqa: E402
    classify_missingness,
    config_hash,
    core_celltypes,
    derive_patient_outcomes,
    extract_patient_endpoints,
    load_config,
    log_jsonl,
    parse_cellpair_columns,
    sha256_file,
)

CFG = load_config(CONFIG_PATH)
CFG_HASH = config_hash(CFG)

OUTPUT_ROOT = Path(CFG["output_root"])
INPUT_ROOT = OUTPUT_ROOT / "inputs"
PATIENT_INPUT_ROOT = INPUT_ROOT / "per_patient"
PATIENT_RUN_ROOT = OUTPUT_ROOT / "cpdb_runs" / "per_patient"
ALL_INPUT_DIR = INPUT_ROOT / "all"
PRIMARY_DIR = OUTPUT_ROOT / "primary"
LOG_DIR = OUTPUT_ROOT / "logs"

RUN_LOG = LOG_DIR / "03_run_log.jsonl"
SUCCESS_RECORD = LOG_DIR / "03_success_record.json"

OUT_SCORES_LONG = PRIMARY_DIR / "patient_endpoint_scores_long.parquet"
OUT_AVAILABILITY = PRIMARY_DIR / "patient_endpoint_availability.parquet"
OUT_METADATA = PRIMARY_DIR / "endpoint_metadata.parquet"
OUT_QC = PRIMARY_DIR / "endpoint_qc.csv"
OUT_SECONDARY = PRIMARY_DIR / "patient_secondary_endpoint_scores.parquet"
OUT_SECONDARY_SPEC = PRIMARY_DIR / "secondary_family_definition.json"
OUT_WIDE = PRIMARY_DIR / "patient_endpoint_scores_wide.parquet"
OUT_PATIENT_OUTCOMES = PRIMARY_DIR / "patient_outcomes.csv"

SOURCE_TABLE = "interaction_scores.csv"

CELLPAIR_SEPARATOR = "|"

# CellPhoneDB metadata columns to preserve verbatim (CellPhoneDB 5.0.1 layout).
META_COLUMNS = (
    "id_cp_interaction", "interacting_pair", "partner_a", "partner_b", "gene_a", "gene_b",
    "secreted", "receptor_a", "receptor_b", "annotation_strategy", "is_integrin",
    "directionality", "classification",
)
KEY_COLUMN = "id_cp_interaction"
ANNOTATION_COLUMNS = tuple(c for c in META_COLUMNS if c != KEY_COLUMN)

STATUS_LEVELS = ("evaluated", "structural_missing", "unexpected_missing", "invalid")
REASON_LEVELS = (
    "computed",
    "sender_not_retained",
    "receiver_not_retained",
    "sender_and_receiver_not_retained",
    "interaction_row_absent_for_patient",
    "non_finite_score",
    "duplicate_endpoint_key",
)

# Plan expectations. Asserted, never forced.
EXPECTED_N_PATIENTS = 24
EXPECTED_N_CELLTYPES = 19
SECONDARY_MIN_PATIENTS = int(CFG["secondary_core_min_patients"])   # 22 of 24
EXPECTED_N_CORE_CELLTYPES = 6
EXPECTED_N_CORE_PAIRS = EXPECTED_N_CORE_CELLTYPES ** 2            # 36 directional pairs
EXPECTED_N_SECONDARY_EVALUABLE_PATIENTS = 20
EXPECTED_SECONDARY_EVALUABLE_SPLIT = {"Good": 11, "Poor": 9}

# Memory guards. The availability grid is |interactions| x |pairs| x |patients|.
MAX_AVAILABILITY_ROWS = 40_000_000
MAX_WIDE_CELLS = 400_000_000

SAMPLE_COLUMN = CFG["sample_column"]
OUTCOME_COLUMN = CFG["outcome_column"]
OUTCOMES = tuple(CFG["outcomes"])

PRIMARY_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

## 2. Shared-API contract check

Every shared helper's positional signature is pinned, so a change in `utils.py` stops this
notebook instead of silently rebinding an argument.

In [ ]:
# ============================================================
# API contract — pin the shared helpers' signatures.
# ============================================================
EXPECTED_SIGNATURES = {
    "load_config": ["path"],
    "config_hash": ["cfg"],
    "sha256_file": ["path", "chunk_size"],
    "log_jsonl": ["path", "record"],
    "parse_cellpair_columns": ["cols", "known_celltypes"],
    "extract_patient_endpoints": ["scores_csv", "retained", "known_celltypes",
                                  "patient", "restrict_to", "id_col"],
    "classify_missingness": ["long_df", "retained_by_patient"],
    "core_celltypes": ["abundance", "min_cells", "min_patients"],
    "derive_patient_outcomes": ["obs", "sample_col", "outcome_col"],
}
UTILS_SIGNATURES = {}
for name, expected in EXPECTED_SIGNATURES.items():
    fn = getattr(utils, name, None)
    assert callable(fn), f"utils.py does not expose {name}()"
    actual = list(inspect.signature(fn).parameters)
    assert actual == expected, (
        f"utils.{name}{inspect.signature(fn)} no longer matches the pinned contract "
        f"{expected}. Update this notebook deliberately; do not adapt silently."
    )
    UTILS_SIGNATURES[name] = str(inspect.signature(fn))

assert utils.PAIR_SEP == CELLPAIR_SEPARATOR, (
    f"utils.PAIR_SEP is {utils.PAIR_SEP!r} but this notebook assumes {CELLPAIR_SEPARATOR!r}"
)
assert tuple(utils.MISSINGNESS_STATUSES) == STATUS_LEVELS, (
    f"utils.MISSINGNESS_STATUSES {utils.MISSINGNESS_STATUSES} != {STATUS_LEVELS}"
)
KEY_SEP, DIR_SEP = utils.KEY_SEP, utils.DIR_SEP
MARKER_NAME = utils.SUCCESS_FILENAME


def parse_pairs(columns, known_celltypes):
    '''utils.parse_cellpair_columns with the arity invariant made explicit.'''
    pairs = parse_cellpair_columns(list(columns), list(known_celltypes))
    assert len(pairs) == len(columns), (
        f"parse_cellpair_columns returned {len(pairs)} pairs for {len(columns)} columns"
    )
    return pairs


def publish_file(tmp_path: Path, final_path: Path) -> Path:
    '''Atomically move a single finished file into place.

    utils.atomic_publish is directory-only (it raises NotADirectoryError on a file), so
    single-file outputs use os.replace, which is atomic within one filesystem. Writing to a
    sibling temporary path keeps a reader from ever seeing a half-written table.
    '''
    tmp_path, final_path = Path(tmp_path), Path(final_path)
    assert tmp_path.is_file() and tmp_path.stat().st_size > 0, f"nothing to publish at {tmp_path}"
    assert tmp_path.parent == final_path.parent, "temp file must be a sibling of its destination"
    final_path.parent.mkdir(parents=True, exist_ok=True)
    os.replace(tmp_path, final_path)
    assert final_path.is_file() and not tmp_path.exists()
    return final_path


UTILS_SIGNATURES

## 3. Patient inventory and the known cell-type vocabulary

Patients are derived from the input tree. The 19 known `annot_6` labels come from the
all-samples input, which by construction retains every label — nothing is hardcoded.

In [ ]:
# ============================================================
# Patients and the label vocabulary that validates cell-pair parsing.
# ============================================================
def read_retained_json(path: Path) -> dict:
    '''Read the retained_celltypes.json written by 01_ (a dict, not a bare list).'''
    payload = json.loads(path.read_text())
    assert isinstance(payload, dict), f"{path}: expected a JSON object, got {type(payload)}"
    for key in ("retained_celltypes", "cells_per_celltype"):
        assert key in payload, f"{path}: missing {key!r}"
    return payload


def read_retained(path: Path):
    labels = sorted(str(x) for x in read_retained_json(path)["retained_celltypes"])
    assert labels, f"No retained cell types in {path}"
    assert len(set(labels)) == len(labels), f"Duplicate cell types in {path}"
    return labels


KNOWN_CELLTYPES = read_retained(ALL_INPUT_DIR / "retained_celltypes.json")
assert len(KNOWN_CELLTYPES) == EXPECTED_N_CELLTYPES, (
    f"{len(KNOWN_CELLTYPES)} labels in the all-samples input, expected {EXPECTED_N_CELLTYPES}"
)
# Splitting on '|' is only safe because no label contains it. Labels legitimately contain
# ' - ' and '/' (e.g. 'Macrophages - AP/TAM'), which is exactly why the delimiter is validated
# rather than assumed.
bad_labels = [c for c in KNOWN_CELLTYPES if CELLPAIR_SEPARATOR in c]
assert not bad_labels, (
    f"Cell-type labels contain the pair separator {CELLPAIR_SEPARATOR!r}: {bad_labels}. "
    "Cell-pair column parsing is unsafe; stop and choose a different delimiter."
)

PATIENTS = sorted(p.name for p in PATIENT_INPUT_ROOT.iterdir() if p.is_dir())
assert len(PATIENTS) == EXPECTED_N_PATIENTS, (
    f"{len(PATIENTS)} per-patient input directories, expected {EXPECTED_N_PATIENTS}"
)

RETAINED, CELLS_PER_TYPE = {}, {}
for p in PATIENTS:
    payload = read_retained_json(PATIENT_INPUT_ROOT / p / "retained_celltypes.json")
    labels = sorted(str(x) for x in payload["retained_celltypes"])
    unknown = sorted(set(labels) - set(KNOWN_CELLTYPES))
    assert not unknown, f"{p}: retained cell types outside the known vocabulary: {unknown}"
    assert len(labels) >= 2, f"{p}: fewer than 2 retained cell types"
    RETAINED[p] = labels
    CELLS_PER_TYPE[p] = {str(k): int(v) for k, v in payload["cells_per_celltype"].items()}
    # The retained list and the recorded counts must agree on the abundance rule.
    from_counts = sorted(t for t, n in CELLS_PER_TYPE[p].items()
                         if n >= int(CFG["patient_celltype_min_cells"]))
    assert from_counts == labels, (
        f"{p}: retained_celltypes {labels} disagrees with cells_per_celltype at the "
        f"{CFG['patient_celltype_min_cells']}-cell threshold ({from_counts})"
    )

# Every patient must have a verified 02_ run before any endpoint is read.
SOURCE_FILES = {}
for p in PATIENTS:
    run_dir = PATIENT_RUN_ROOT / p
    marker = run_dir / MARKER_NAME
    assert marker.is_file(), f"{p}: no completion marker at {marker}; rerun 02_"
    m = json.loads(marker.read_text())
    assert m.get("status") == "success", f"{p}: marker status {m.get('status')!r}"
    assert m.get("config_hash") == CFG_HASH, f"{p}: run made under a different configuration"
    assert sorted(m.get("retained_celltypes", [])) == RETAINED[p], (
        f"{p}: run marker cell types disagree with retained_celltypes.json"
    )
    src = run_dir / SOURCE_TABLE
    assert src.is_file() and src.stat().st_size > 0, f"{p}: missing {SOURCE_TABLE}"
    recorded = (m.get("output_hashes") or {}).get(SOURCE_TABLE)
    assert recorded is not None, f"{p}: marker records no hash for {SOURCE_TABLE}"
    assert sha256_file(src) == recorded, f"{p}: {SOURCE_TABLE} changed since 02_ published it"
    SOURCE_FILES[p] = src

pd.DataFrame({"patient": PATIENTS,
              "n_retained": [len(RETAINED[p]) for p in PATIENTS],
              "n_pairs": [len(RETAINED[p]) ** 2 for p in PATIENTS]}).set_index("patient").T

## 4. Per-patient extraction

Schema is detected from the column names of each file; nothing about the layout is assumed
beyond the CellPhoneDB 5.0.1 metadata block. `utils.extract_patient_endpoints` does the
melt; this notebook validates the result against invariants that must hold for the extraction
to be trustworthy.

In [ ]:
# ============================================================
# Read one patient's interaction_scores.csv into long form.
# ============================================================
def load_patient(patient: str):
    path = SOURCE_FILES[patient]
    retained = RETAINED[patient]
    df = pd.read_csv(path)

    # --- schema detection -------------------------------------------------
    missing_meta = [c for c in META_COLUMNS if c not in df.columns]
    assert not missing_meta, f"{patient}: {SOURCE_TABLE} missing metadata columns {missing_meta}"
    pair_cols = [c for c in df.columns if CELLPAIR_SEPARATOR in c]
    unclassified = sorted(set(df.columns) - set(META_COLUMNS) - set(pair_cols))
    assert not unclassified, (
        f"{patient}: columns that are neither metadata nor cell pairs: {unclassified}"
    )
    assert pair_cols, f"{patient}: no cell-pair columns"

    # --- delimiter-safe parsing, validated against the known vocabulary ----
    pairs = parse_pairs(pair_cols, retained)
    allowed = set(retained)
    offenders = sorted({t for pair in pairs for t in pair if t not in allowed})
    assert not offenders, (
        f"{patient}: cell-pair columns reference cell types not retained for this patient: "
        f"{offenders}. Either the delimiter split wrongly or 02_ ran on different inputs."
    )
    expected_pairs = {(a, b) for a in retained for b in retained}
    assert set(pairs) == expected_pairs, (
        f"{patient}: cell-pair columns are not the full {len(retained)}^2 grid "
        f"({len(pairs)} present, {len(expected_pairs)} expected)"
    )

    # --- endpoint keys ----------------------------------------------------
    dup = df[KEY_COLUMN][df[KEY_COLUMN].duplicated()].unique().tolist()
    assert not dup, (
        f"{patient}: duplicate {KEY_COLUMN} in {SOURCE_TABLE}: {dup[:10]}. "
        "Endpoint keys would not be unique; stop and inspect the CellPhoneDB output."
    )

    metadata = df[list(META_COLUMNS)].copy()
    max_score = float(df[pair_cols].apply(pd.to_numeric, errors="coerce").to_numpy().max())

    # --- melt to long, via the shared helper (it takes the CSV path) -------
    long = extract_patient_endpoints(path, retained, KNOWN_CELLTYPES, patient=patient)
    assert isinstance(long, pd.DataFrame)
    for col in ("patient", "id_cp_interaction", "sender", "receiver", "score",
                "score_present", "non_numeric", "duplicate_id"):
        assert col in long.columns, f"extract_patient_endpoints output lacks {col!r}"

    # --- invariants: the melt must be lossless and rectangular -------------
    n_expected = len(df) * len(pair_cols)
    assert len(long) == n_expected, (
        f"{patient}: long form has {len(long)} rows, expected {len(df)} interactions x "
        f"{len(pair_cols)} pairs = {n_expected}. Because the CSV grid and the retained x "
        "retained grid coincide, the union the helper builds must be exactly this size."
    )
    assert not long.duplicated(["id_cp_interaction", "sender", "receiver"]).any(), (
        f"{patient}: duplicate (id_cp_interaction, sender, receiver) after melting"
    )
    assert bool(long["score_present"].all()), (
        f"{patient}: {int((~long['score_present']).sum())} endpoints in the expected grid have "
        "no column in the CSV"
    )
    assert int(long["non_numeric"].sum()) == 0, f"{patient}: non-numeric scores present"
    assert int(long["duplicate_id"].sum()) == 0, f"{patient}: duplicate id_cp_interaction"
    assert bool(long["sender_retained"].all()) and bool(long["receiver_retained"].all())
    assert long["score"].notna().all(), (
        f"{patient}: {int(long['score'].isna().sum())} NaN scores inside the published grid. "
        "CellPhoneDB fills unmatched ligand-receptor products with 0, so NaN here is "
        "unexplained; stop and inspect."
    )
    assert np.isfinite(long["score"].to_numpy()).all(), f"{patient}: non-finite scores"
    # Direction must survive: A|B and B|A are distinct columns and stay distinct rows.
    assert (long["sender"] != long["receiver"]).any(), f"{patient}: no cross-type pairs survived"

    long = long[["patient", "id_cp_interaction", "sender", "receiver", "score"]].copy()
    return metadata, long, len(df), len(pair_cols), max_score

In [ ]:
# ============================================================
# Run the extraction for all 24 patients.
# ============================================================
started = datetime.now(timezone.utc).isoformat()
long_parts, meta_parts, patient_rows = [], [], []

for i, patient in enumerate(PATIENTS, start=1):
    metadata, long, n_interactions, n_pairs, max_score = load_patient(patient)
    long_parts.append(long)
    meta_parts.append(metadata)
    patient_rows.append({
        "patient": patient,
        "n_retained_celltypes": len(RETAINED[patient]),
        "n_cell_pairs": n_pairs,
        "n_interactions": n_interactions,
        "n_evaluated": len(long),
        "n_nonzero": int((long["score"] != 0).sum()),
        "max_score": max_score,
    })
    record = {"event": "patient_extracted", **patient_rows[-1],
              "index": i, "timestamp_utc": datetime.now(timezone.utc).isoformat()}
    log_jsonl(RUN_LOG, record)

PATIENT_SUMMARY = pd.DataFrame(patient_rows).set_index("patient")
PATIENT_SUMMARY["nonzero_density"] = (
    PATIENT_SUMMARY["n_nonzero"] / PATIENT_SUMMARY["n_evaluated"]
)

# Row counts genuinely differ between patients, which is why a long union is mandatory:
# there is no common wide schema to align on.
assert PATIENT_SUMMARY["n_interactions"].nunique() > 1, (
    "Every patient has the same interaction count — unexpected for per-sample CellPhoneDB "
    "runs; verify that 02_ did not reuse one patient's output for all patients."
)
PATIENT_SUMMARY

In [ ]:
# ============================================================
# Efficient dtypes: one categorical dictionary shared by every patient's rows.
# ============================================================
SCORES = pd.concat(long_parts, ignore_index=True, copy=False)
del long_parts
gc.collect()

INTERACTIONS = sorted(SCORES["id_cp_interaction"].unique())
OBSERVED_PAIRS = sorted(
    {(s, r) for s, r in zip(SCORES["sender"], SCORES["receiver"])}
)
SENDERS = sorted({s for s, _ in OBSERVED_PAIRS})
RECEIVERS = sorted({r for _, r in OBSERVED_PAIRS})
PAIR_TYPES = sorted(set(SENDERS) | set(RECEIVERS))

patient_dtype = pd.CategoricalDtype(PATIENTS, ordered=False)
interaction_dtype = pd.CategoricalDtype(INTERACTIONS, ordered=False)
celltype_dtype = pd.CategoricalDtype(PAIR_TYPES, ordered=False)

SCORES["patient"] = SCORES["patient"].astype(patient_dtype)
SCORES["id_cp_interaction"] = SCORES["id_cp_interaction"].astype(interaction_dtype)
SCORES["sender"] = SCORES["sender"].astype(celltype_dtype)
SCORES["receiver"] = SCORES["receiver"].astype(celltype_dtype)
SCORES["score"] = SCORES["score"].astype("float64")

assert not SCORES.duplicated(["patient", "id_cp_interaction", "sender", "receiver"]).any(), (
    "Duplicate (patient, endpoint) rows in the union"
)
assert set(PAIR_TYPES) <= set(KNOWN_CELLTYPES), "Unknown cell types leaked into the union"

# The universe over which missingness is defined: every interaction seen in any patient x
# every directional pair seen in any patient.
N_INTERACTIONS = len(INTERACTIONS)
N_PAIRS = len(OBSERVED_PAIRS)
N_UNIVERSE = N_INTERACTIONS * N_PAIRS
print(f"patients                : {len(PATIENTS)}")
print(f"union interactions      : {N_INTERACTIONS}")
print(f"union directional pairs : {N_PAIRS} (over {len(PAIR_TYPES)} cell types)")
print(f"endpoint universe       : {N_UNIVERSE:,}")
print(f"evaluated rows          : {len(SCORES):,}")
print(f"availability grid rows  : {N_UNIVERSE * len(PATIENTS):,}")
assert N_UNIVERSE * len(PATIENTS) <= MAX_AVAILABILITY_ROWS, (
    f"Availability grid of {N_UNIVERSE * len(PATIENTS):,} rows exceeds the "
    f"{MAX_AVAILABILITY_ROWS:,} guard; raise the guard deliberately or store it factorised."
)

## 5. Endpoint metadata

The 12 CellPhoneDB annotation columns must be identical for a given `id_cp_interaction` in
every patient. If they are not, the files are not describing the same interaction and nothing
downstream is safe.

In [ ]:
# ============================================================
# One metadata row per id_cp_interaction, verified consistent across patients.
# ============================================================
META_ALL = pd.concat(meta_parts, ignore_index=True, copy=False)
del meta_parts
gc.collect()

ENDPOINT_METADATA = META_ALL.drop_duplicates().reset_index(drop=True)
conflicting = ENDPOINT_METADATA[KEY_COLUMN][ENDPOINT_METADATA[KEY_COLUMN].duplicated()].unique()
if len(conflicting):
    detail = (META_ALL[META_ALL[KEY_COLUMN].isin(conflicting[:5])]
              .drop_duplicates()
              .sort_values(KEY_COLUMN))
    raise AssertionError(
        f"{len(conflicting)} interaction(s) carry inconsistent metadata across patients. "
        f"Examples:\n{detail.to_string(index=False)}"
    )

ENDPOINT_METADATA = ENDPOINT_METADATA.set_index(KEY_COLUMN).loc[INTERACTIONS].reset_index()
assert len(ENDPOINT_METADATA) == N_INTERACTIONS
assert list(ENDPOINT_METADATA.columns) == list(META_COLUMNS)
assert ENDPOINT_METADATA[KEY_COLUMN].is_unique
assert ENDPOINT_METADATA["interacting_pair"].notna().all()
del META_ALL
gc.collect()
ENDPOINT_METADATA.head()

## 6. Availability and missingness classification

For every patient and every endpoint in the universe:

* `structural_missing` — sender or receiver not retained for that patient (reconciles exactly
  against `retained_celltypes.json`);
* `unexpected_missing` — pair available but the interaction row is absent from that patient's
  CellPhoneDB output;
* `evaluated` — a finite computed score, **zero included**;
* `invalid` — non-finite or duplicated.

Built and written one patient at a time so the full grid is never held in memory twice.

In [ ]:
# ============================================================
# Per-patient availability, written incrementally to Parquet.
# ============================================================
status_dtype = pd.CategoricalDtype(STATUS_LEVELS, ordered=False)
reason_dtype = pd.CategoricalDtype(REASON_LEVELS, ordered=False)
REASON_CODE = {s: i for i, s in enumerate(REASON_LEVELS)}

UNIVERSE_SENDER = np.array([s for s, _ in OBSERVED_PAIRS])
UNIVERSE_RECEIVER = np.array([r for _, r in OBSERVED_PAIRS])
CELLTYPE_CODE = {c: i for i, c in enumerate(PAIR_TYPES)}
SENDER_CODES = np.array([CELLTYPE_CODE[s] for s in UNIVERSE_SENDER], dtype=np.int16)
RECEIVER_CODES = np.array([CELLTYPE_CODE[r] for r in UNIVERSE_RECEIVER], dtype=np.int16)
PATIENT_CODE = {p: i for i, p in enumerate(PATIENTS)}

# Dense (sender_code, receiver_code) -> position in OBSERVED_PAIRS, for O(1) scatter.
PAIR_POS = np.full((len(PAIR_TYPES), len(PAIR_TYPES)), -1, dtype=np.int32)
for _k, (_s, _r) in enumerate(OBSERVED_PAIRS):
    PAIR_POS[CELLTYPE_CODE[_s], CELLTYPE_CODE[_r]] = _k

INTERACTION_INDEX = pd.Index(INTERACTIONS)
PATIENT_INTERACTIONS = {
    p: set(g["id_cp_interaction"].cat.remove_unused_categories().cat.categories)
    for p, g in SCORES.groupby("patient", observed=True)
}
assert set(PATIENT_INTERACTIONS) == set(PATIENTS)

# The universe is laid out interaction-major: interaction i occupies rows [i*N_PAIRS, ...).
UNIVERSE_ID_CODES = np.repeat(np.arange(N_INTERACTIONS, dtype=np.int32), N_PAIRS)
UNIVERSE_SENDER_CODES = np.tile(SENDER_CODES, N_INTERACTIONS)
UNIVERSE_RECEIVER_CODES = np.tile(RECEIVER_CODES, N_INTERACTIONS)
_pair_keys = np.array([f"{s}{DIR_SEP}{r}" for s, r in OBSERVED_PAIRS])
UNIVERSE_ENDPOINT_KEY = np.char.add(
    np.char.add(np.repeat(np.array(INTERACTIONS, dtype=object), N_PAIRS).astype(str), KEY_SEP),
    np.tile(_pair_keys, N_INTERACTIONS),
)
assert len(UNIVERSE_ENDPOINT_KEY) == N_UNIVERSE


def patient_universe_frame(patient: str):
    '''One patient's full (interaction x pair) grid, with observed scores scattered in.

    Rows the CellPhoneDB output did not contain are left NaN and score_present=False. They
    are NOT zero-filled: classify_missingness decides what their absence means.
    '''
    blk = SCORES.loc[SCORES["patient"] == patient]
    row = blk["id_cp_interaction"].cat.codes.to_numpy()
    col = PAIR_POS[blk["sender"].cat.codes.to_numpy(), blk["receiver"].cat.codes.to_numpy()]
    assert (col >= 0).all(), f"{patient}: an observed cell pair is not in the universe"

    score_mat = np.full((N_INTERACTIONS, N_PAIRS), np.nan, dtype="float64")
    present_mat = np.zeros((N_INTERACTIONS, N_PAIRS), dtype=bool)
    score_mat[row, col] = blk["score"].to_numpy()
    present_mat[row, col] = True
    assert int(present_mat.sum()) == len(blk), f"{patient}: scatter lost rows"

    score_flat = score_mat.ravel()
    present_flat = present_mat.ravel()
    return pd.DataFrame({
        "patient": pd.Categorical.from_codes(
            np.full(N_UNIVERSE, PATIENT_CODE[patient], dtype=np.int8), dtype=patient_dtype),
        "id_cp_interaction": pd.Categorical.from_codes(UNIVERSE_ID_CODES, dtype=interaction_dtype),
        "sender": pd.Categorical.from_codes(UNIVERSE_SENDER_CODES, dtype=celltype_dtype),
        "receiver": pd.Categorical.from_codes(UNIVERSE_RECEIVER_CODES, dtype=celltype_dtype),
        "endpoint_key": UNIVERSE_ENDPOINT_KEY,
        "score_raw": score_flat,
        "score": score_flat,
        "score_present": present_flat,
        "non_numeric": np.zeros(N_UNIVERSE, dtype=bool),
        "duplicate_id": np.zeros(N_UNIVERSE, dtype=bool),
    })


def availability_reason(frame: pd.DataFrame):
    '''Supplementary, finer-grained explanation for each shared status value.'''
    sender_ok = frame["sender_retained"].to_numpy(dtype=bool)
    receiver_ok = frame["receiver_retained"].to_numpy(dtype=bool)
    present = frame["score_present"].to_numpy(dtype=bool)
    codes = np.full(len(frame), REASON_CODE["computed"], dtype=np.int8)
    codes[~sender_ok & receiver_ok] = REASON_CODE["sender_not_retained"]
    codes[sender_ok & ~receiver_ok] = REASON_CODE["receiver_not_retained"]
    codes[~sender_ok & ~receiver_ok] = REASON_CODE["sender_and_receiver_not_retained"]
    codes[sender_ok & receiver_ok & ~present] = REASON_CODE["interaction_row_absent_for_patient"]
    return pd.Categorical.from_codes(codes, dtype=reason_dtype)

In [ ]:
# ============================================================
# Build availability, reconcile exactly, and write it.
# ============================================================
tmp_availability = OUT_AVAILABILITY.with_suffix(".parquet.tmp")
if tmp_availability.exists():
    tmp_availability.unlink()

availability_counts, reconciliation = {}, []
writer = None
try:
    for patient in PATIENTS:
        # utils.classify_missingness is the single definition of the four statuses; this
        # notebook only supplies the grid and checks the arithmetic it produces.
        frame = classify_missingness(patient_universe_frame(patient), RETAINED)
        frame["reason"] = availability_reason(frame)

        n_present = len(PATIENT_INTERACTIONS[patient])

        # The two meanings must stay apart: a non-evaluated endpoint carries no number at all,
        # and every score CellPhoneDB computed -- zeros included -- survives untouched.
        evaluated = (frame["status"] == "evaluated").to_numpy()
        assert frame.loc[~evaluated, "score_evaluated"].isna().all(), (
            f"{patient}: a structurally missing or absent endpoint carries a score"
        )
        assert frame.loc[evaluated, "score_evaluated"].notna().all()
        observed = SCORES.loc[SCORES["patient"] == patient, "score"]
        assert int((frame.loc[evaluated, "score_evaluated"] == 0).sum()) == int((observed == 0).sum()), (
            f"{patient}: the number of computed zeros changed between the score table and the "
            "availability grid -- zeros are being confused with missingness"
        )
        assert np.isclose(float(frame.loc[evaluated, "score_evaluated"].sum()),
                          float(observed.sum())), f"{patient}: scores altered during scatter"

        frame = frame[["patient", "id_cp_interaction", "sender", "receiver",
                       "endpoint_key", "status", "reason", "score_evaluated"]]
        counts = frame["status"].value_counts().reindex(STATUS_LEVELS, fill_value=0)
        availability_counts[patient] = counts.to_dict()

        # Exact reconciliation against retained_celltypes.json — arithmetic, not tolerance.
        n_retained = len(RETAINED[patient])
        n_avail_pairs = sum(1 for s, r in OBSERVED_PAIRS
                            if s in set(RETAINED[patient]) and r in set(RETAINED[patient]))
        assert n_avail_pairs == n_retained ** 2, (
            f"{patient}: {n_avail_pairs} of this patient's pairs appear in the universe, "
            f"expected {n_retained}^2 = {n_retained ** 2}"
        )
        exp_structural = N_INTERACTIONS * (N_PAIRS - n_avail_pairs)
        exp_evaluated = n_present * n_avail_pairs
        exp_unexpected = (N_INTERACTIONS - n_present) * n_avail_pairs
        assert counts["structural_missing"] == exp_structural, (
            f"{patient}: structural_missing {counts['structural_missing']} != "
            f"{N_INTERACTIONS} interactions x ({N_PAIRS} - {n_avail_pairs}) pairs = {exp_structural}"
        )
        assert counts["evaluated"] == exp_evaluated, (
            f"{patient}: evaluated {counts['evaluated']} != {exp_evaluated}"
        )
        assert counts["unexpected_missing"] == exp_unexpected, (
            f"{patient}: unexpected_missing {counts['unexpected_missing']} != {exp_unexpected}"
        )
        assert counts["invalid"] == 0, f"{patient}: {counts['invalid']} invalid endpoints"
        assert counts.sum() == N_INTERACTIONS * N_PAIRS

        reconciliation.append({
            "patient": patient, "n_retained_celltypes": n_retained,
            "n_available_pairs": n_avail_pairs, "n_interactions_present": n_present,
            "expected_structural_missing": exp_structural,
            "expected_evaluated": exp_evaluated,
            "expected_unexpected_missing": exp_unexpected,
            **counts.to_dict(),
        })

        table = pa.Table.from_pandas(frame, preserve_index=False)
        if writer is None:
            writer = pq.ParquetWriter(tmp_availability, table.schema, compression="zstd")
        writer.write_table(table)
        del frame, table
        gc.collect()
finally:
    if writer is not None:
        writer.close()

RECONCILIATION = pd.DataFrame(reconciliation).set_index("patient")
AVAILABILITY_COUNTS = pd.DataFrame(availability_counts).T[list(STATUS_LEVELS)]
AVAILABILITY_COUNTS.sum()

In [ ]:
# ============================================================
# The evaluated cells in the availability grid must be exactly the rows we actually read.
# ============================================================
assert int(AVAILABILITY_COUNTS["evaluated"].sum()) == len(SCORES), (
    f"availability marks {int(AVAILABILITY_COUNTS['evaluated'].sum()):,} evaluated endpoints "
    f"but the score table holds {len(SCORES):,}"
)
assert int(AVAILABILITY_COUNTS.to_numpy().sum()) == N_UNIVERSE * len(PATIENTS)

# unexpected_missing must be row-structured: an interaction absent for a patient is absent for
# ALL of that patient's available pairs, never for some. This is the falsifiable form of
# "CellPhoneDB dropped the interaction row for this sample".
for _, row in RECONCILIATION.iterrows():
    assert row["unexpected_missing"] % row["n_available_pairs"] == 0, (
        f"{row.name}: unexpected_missing is not a whole multiple of the available-pair count, "
        "so missingness is scattered rather than row-structured. Stop and inspect."
    )
    assert (row["unexpected_missing"] // row["n_available_pairs"]
            == N_INTERACTIONS - row["n_interactions_present"])

publish_file(tmp_availability, OUT_AVAILABILITY)
assert OUT_AVAILABILITY.is_file() and not tmp_availability.exists()
RECONCILIATION

## 7. Write the authoritative long score table

In [ ]:
# ============================================================
# patient_endpoint_scores_long.parquet — evaluated endpoints only, zeros preserved as zeros.
# ============================================================
SCORES = SCORES.sort_values(["id_cp_interaction", "sender", "receiver", "patient"],
                            kind="stable").reset_index(drop=True)
SCORES["status"] = pd.Categorical(["evaluated"] * len(SCORES), dtype=status_dtype)

tmp_scores = OUT_SCORES_LONG.with_suffix(".parquet.tmp")
pq.write_table(pa.Table.from_pandas(SCORES, preserve_index=False), tmp_scores, compression="zstd")
publish_file(tmp_scores, OUT_SCORES_LONG)
assert OUT_SCORES_LONG.is_file() and not tmp_scores.exists()

tmp_meta = OUT_METADATA.with_suffix(".parquet.tmp")
pq.write_table(pa.Table.from_pandas(ENDPOINT_METADATA, preserve_index=False), tmp_meta,
               compression="zstd")
publish_file(tmp_meta, OUT_METADATA)
assert OUT_METADATA.is_file() and not tmp_meta.exists()

len(SCORES), OUT_SCORES_LONG.stat().st_size / 1e6

## 8. Endpoint QC

One row per endpoint key, with no outcome information anywhere in it.

In [ ]:
# ============================================================
# endpoint_qc.csv — per-endpoint evaluable / missing / non-zero counts.
# ============================================================
SCORES["is_nonzero"] = (SCORES["score"] != 0).astype("int8")
QC = (SCORES.groupby(["id_cp_interaction", "sender", "receiver"], observed=True)
      .agg(n_evaluated=("score", "size"),
           n_nonzero=("is_nonzero", "sum"),
           median_score=("score", "median"),
           max_score=("score", "max"))
      .reset_index())

# Availability per (sender, receiver): how many patients could the endpoint exist for at all.
pair_n_available = {
    (s, r): sum(1 for p in PATIENTS if s in set(RETAINED[p]) and r in set(RETAINED[p]))
    for s, r in OBSERVED_PAIRS
}
QC["n_patients_pair_available"] = [
    pair_n_available[(s, r)] for s, r in zip(QC["sender"], QC["receiver"])
]
QC["n_structural_missing"] = len(PATIENTS) - QC["n_patients_pair_available"]
QC["n_unexpected_missing"] = QC["n_patients_pair_available"] - QC["n_evaluated"]
QC["n_invalid"] = 0

assert not QC.duplicated(["id_cp_interaction", "sender", "receiver"]).any(), (
    "Duplicated endpoint key in the QC table"
)
assert (QC["n_evaluated"] + QC["n_unexpected_missing"] + QC["n_structural_missing"]
        == len(PATIENTS)).all(), "Per-endpoint statuses do not sum to 24 patients"
assert (QC["n_unexpected_missing"] >= 0).all(), (
    "An endpoint was evaluated in more patients than the pair was available in"
)
assert (QC["n_nonzero"] <= QC["n_evaluated"]).all()

tmp_qc = OUT_QC.with_suffix(".csv.tmp")
QC.to_csv(tmp_qc, index=False)
publish_file(tmp_qc, OUT_QC)
assert OUT_QC.is_file() and not tmp_qc.exists()

QC[["n_evaluated", "n_structural_missing", "n_unexpected_missing", "n_nonzero"]].describe()

## 9. Gate 3

1. No duplicated endpoint key anywhere.
2. Structural-missing counts reconcile **exactly** against `retained_celltypes.json`.
3. Zero `unexpected_missing`, or every instance explained by one documented CellPhoneDB
   mechanism.

In [ ]:
# ============================================================
# GATE 3.1 — endpoint keys are unique.
# ============================================================
assert not SCORES.duplicated(["patient", "id_cp_interaction", "sender", "receiver"]).any()
assert not QC.duplicated(["id_cp_interaction", "sender", "receiver"]).any()
assert ENDPOINT_METADATA[KEY_COLUMN].is_unique

# GATE 3.2 — structural missingness reconciles arithmetically, patient by patient.
for patient in PATIENTS:
    retained = set(RETAINED[patient])
    n_avail = sum(1 for s, r in OBSERVED_PAIRS if s in retained and r in retained)
    assert n_avail == len(retained) ** 2
    expected = N_INTERACTIONS * (N_PAIRS - n_avail)
    got = int(AVAILABILITY_COUNTS.loc[patient, "structural_missing"])
    assert got == expected, f"{patient}: structural_missing {got} != {expected}"

# Cross-check from the other direction: every endpoint the patient is missing for must involve
# a cell type that patient genuinely lacks.
for patient in PATIENTS:
    retained = set(RETAINED[patient])
    evaluated_pairs = {
        (s, r) for s, r in zip(
            SCORES.loc[SCORES["patient"] == patient, "sender"],
            SCORES.loc[SCORES["patient"] == patient, "receiver"])
    }
    assert all(s in retained and r in retained for s, r in evaluated_pairs), (
        f"{patient}: an evaluated endpoint uses a cell type absent from retained_celltypes.json"
    )
    assert evaluated_pairs == {(s, r) for s in retained for r in retained}, (
        f"{patient}: evaluated pairs are not the full retained grid"
    )
print("GATE 3.1-3.2 passed")

In [ ]:
# ============================================================
# GATE 3.3 — every unexpected_missing instance is explained, or the notebook stops.
# ============================================================
UNEXPECTED_TOTAL = int(AVAILABILITY_COUNTS["unexpected_missing"].sum())

EXPLANATION = []
for patient in PATIENTS:
    present = PATIENT_INTERACTIONS[patient]
    absent = [i for i in INTERACTIONS if i not in present]
    n_avail = len(RETAINED[patient]) ** 2
    EXPLANATION.append({
        "patient": patient,
        "n_interactions_absent": len(absent),
        "n_available_pairs": n_avail,
        "unexpected_missing": len(absent) * n_avail,
        "explanation": "interaction_row_absent_for_patient",
    })
EXPLANATION = pd.DataFrame(EXPLANATION).set_index("patient")

assert int(EXPLANATION["unexpected_missing"].sum()) == UNEXPECTED_TOTAL, (
    "unexpected_missing cannot be fully accounted for by absent interaction rows"
)
unexplained = UNEXPECTED_TOTAL - int(EXPLANATION["unexpected_missing"].sum())
assert unexplained == 0, f"{unexplained} unexplained unexpected_missing endpoints"

# The mechanism itself: CellPhoneDB emits no row for an interaction when no participant clears
# the expression filter in that sample, so per-patient row counts differ. Verify the absent
# interactions really are absent from the file rather than present-with-NaN.
for patient in PATIENTS[:3] if len(PATIENTS) > 3 else PATIENTS:
    ids_in_file = set(pd.read_csv(SOURCE_FILES[patient], usecols=[KEY_COLUMN])[KEY_COLUMN])
    assert ids_in_file == PATIENT_INTERACTIONS[patient], (
        f"{patient}: interactions parsed from the long table differ from the raw file"
    )
    absent = set(INTERACTIONS) - ids_in_file
    assert not (absent & ids_in_file)

print(f"GATE 3.3 passed — {UNEXPECTED_TOTAL:,} unexpected_missing endpoints, "
      f"100% explained by CellPhoneDB per-sample interaction-row filtering")
EXPLANATION

## 10. Secondary family — a fixed 36-pair core grid

Core cell types are those retained (>= 10 cells) in at least 22 of 24 patients. The secondary
endpoint for one interaction and one patient is the mean score over **all 36** fixed
directional core pairs. A patient is evaluable only when all 36 are computed, so the averaged
quantity is literally the same quantity in every patient — never a mean over whichever pairs
happened to exist.

In [ ]:
# ============================================================
# Core cell types and the fixed 36-pair grid.
# ============================================================
# utils.core_celltypes wants an abundance matrix with rows = cell types, columns = patients.
# It is reconstructed from the cells_per_celltype block 01_ wrote into each patient's
# retained_celltypes.json, so the source h5ad is not needed here. Types below the abundance
# threshold are absent from that block and therefore enter as 0 — which cannot change the
# ">= min_cells" answer, only make a sub-threshold count unrecoverable.
ABUNDANCE = (pd.DataFrame({p: pd.Series(CELLS_PER_TYPE[p]) for p in PATIENTS})
             .reindex(index=KNOWN_CELLTYPES, columns=PATIENTS)
             .fillna(0).astype(int))
ABUNDANCE.index.name = CFG["cell_type_column"]
ABUNDANCE.columns.name = SAMPLE_COLUMN
assert ABUNDANCE.shape == (len(KNOWN_CELLTYPES), len(PATIENTS)), ABUNDANCE.shape

MIN_CELLS = int(CFG["patient_celltype_min_cells"])
CORE = sorted(str(c) for c in core_celltypes(ABUNDANCE, MIN_CELLS, SECONDARY_MIN_PATIENTS))

# Independent recomputation — the shared helper and the retained lists must agree.
retention_counts = Counter(t for p in PATIENTS for t in RETAINED[p])
CORE_CHECK = sorted(t for t, n in retention_counts.items() if n >= SECONDARY_MIN_PATIENTS)
assert CORE == CORE_CHECK, (
    f"utils.core_celltypes returned {CORE} but a direct count over retained_celltypes.json "
    f"gives {CORE_CHECK} at >= {SECONDARY_MIN_PATIENTS}/{len(PATIENTS)} patients"
)
assert len(CORE) == EXPECTED_N_CORE_CELLTYPES, (
    f"{len(CORE)} core cell types at >= {SECONDARY_MIN_PATIENTS}/{len(PATIENTS)} patients, "
    f"plan expects {EXPECTED_N_CORE_CELLTYPES}. Core set: {CORE}"
)

CORE_PAIRS = [(a, b) for a in CORE for b in CORE]
assert len(CORE_PAIRS) == EXPECTED_N_CORE_PAIRS == 36
assert set(CORE_PAIRS) <= set(OBSERVED_PAIRS), (
    "Some core pairs never appear in any patient's output"
)

CORE_COMPLETE_PATIENTS = [p for p in PATIENTS if set(CORE) <= set(RETAINED[p])]
assert len(CORE_COMPLETE_PATIENTS) == EXPECTED_N_SECONDARY_EVALUABLE_PATIENTS, (
    f"{len(CORE_COMPLETE_PATIENTS)} patients retain all {len(CORE)} core cell types, "
    f"plan expects {EXPECTED_N_SECONDARY_EVALUABLE_PATIENTS}. "
    f"Missing: {sorted(set(PATIENTS) - set(CORE_COMPLETE_PATIENTS))}"
)
pd.Series(retention_counts).sort_values(ascending=False).rename("n_patients_retained").to_frame().T

In [ ]:
# ============================================================
# Secondary endpoint values: mean over all 36 core pairs.
# ============================================================
# CORE_PAIRS is the complete CORE x CORE grid, so a pair is core exactly when both ends are
# core cell types. Selected on categorical codes to avoid materialising 2.6M strings.
core_codes = [CELLTYPE_CODE[c] for c in CORE]
core_mask = (SCORES["sender"].cat.codes.isin(core_codes)
             & SCORES["receiver"].cat.codes.isin(core_codes))
CORE_SCORES = SCORES.loc[core_mask, ["patient", "id_cp_interaction", "score"]]
assert set(zip(SCORES.loc[core_mask, "sender"], SCORES.loc[core_mask, "receiver"])) <= set(CORE_PAIRS)

agg = (CORE_SCORES.groupby(["patient", "id_cp_interaction"], observed=True)["score"]
       .agg(n_pairs="size", secondary_score="mean")
       .reset_index())

# Every (patient, interaction) with a complete core grid must have exactly 36 pairs; anything
# else means the fixed-grid guarantee is broken.
complete = agg["n_pairs"] == EXPECTED_N_CORE_PAIRS
partial = agg.loc[~complete]
for _, row in partial.iterrows():
    assert not set(CORE) <= set(RETAINED[str(row["patient"])]), (
        f"{row['patient']}/{row['id_cp_interaction']}: {int(row['n_pairs'])} of "
        f"{EXPECTED_N_CORE_PAIRS} core pairs computed although all core types are retained"
    )

SECONDARY = agg.loc[complete].copy()
SECONDARY["n_pairs"] = SECONDARY["n_pairs"].astype("int16")
assert (SECONDARY["n_pairs"] == EXPECTED_N_CORE_PAIRS).all()
assert set(SECONDARY["patient"].astype(str)) <= set(CORE_COMPLETE_PATIENTS)
assert np.isfinite(SECONDARY["secondary_score"].to_numpy()).all()
assert not SECONDARY.duplicated(["patient", "id_cp_interaction"]).any()

SECONDARY_PATIENTS = sorted(SECONDARY["patient"].astype(str).unique())
assert len(SECONDARY_PATIENTS) == EXPECTED_N_SECONDARY_EVALUABLE_PATIENTS, (
    f"{len(SECONDARY_PATIENTS)} patients contribute secondary endpoints, expected "
    f"{EXPECTED_N_SECONDARY_EVALUABLE_PATIENTS}"
)
print(f"core cell types ({len(CORE)}): {CORE}")
print(f"fixed directional pairs      : {len(CORE_PAIRS)}")
print(f"evaluable patients           : {len(SECONDARY_PATIENTS)} of {len(PATIENTS)}")
print(f"secondary rows               : {len(SECONDARY):,}")

## 11. Optional wide matrix

Non-authoritative. `NaN` in the wide matrix cannot distinguish `structural_missing` from
`unexpected_missing`; the availability table carries that distinction and remains the source
of truth for missingness.

In [ ]:
# ============================================================
# Wide matrix, only if it fits comfortably in memory.
# ============================================================
WIDE_WRITTEN = False
wide_cells = N_UNIVERSE * len(PATIENTS)
if wide_cells <= MAX_WIDE_CELLS:
    WIDE = SCORES.pivot_table(index=["id_cp_interaction", "sender", "receiver"],
                              columns="patient", values="score", observed=True)
    WIDE = WIDE.reindex(columns=PATIENTS).reset_index()
    assert len(WIDE) == len(QC)
    tmp_wide = OUT_WIDE.with_suffix(".parquet.tmp")
    pq.write_table(pa.Table.from_pandas(WIDE, preserve_index=False), tmp_wide, compression="zstd")
    publish_file(tmp_wide, OUT_WIDE)
    WIDE_WRITTEN = True
    del WIDE
    gc.collect()
else:
    print(f"Wide matrix skipped: {wide_cells:,} cells exceeds the {MAX_WIDE_CELLS:,} guard. "
          "The long Parquet table is authoritative.")
WIDE_WRITTEN

## 12. Outcome labels

Joined **only now**, after endpoint construction and all QC are complete, so that nothing in
the endpoint definition, the missingness classification or the QC counts could have been
influenced by outcome.

In [ ]:
# ============================================================
# Patient -> clinical_outcome. Derived, never hardcoded.
# ============================================================
def load_patient_outcomes():
    '''Derive patient -> outcome. Never hardcoded, never inferred from batch_1.

    Preferred source: the `outcomes` block 01_ wrote into each patient's
    retained_celltypes.json, which came from obs[clinical_outcome] for exactly that patient's
    cells. Fallback: utils.derive_patient_outcomes over the source h5ad obs.
    '''
    rows, ok = [], True
    for p in PATIENTS:
        payload = json.loads((PATIENT_INPUT_ROOT / p / "retained_celltypes.json").read_text())
        outcomes = payload.get("outcomes")
        if not (isinstance(outcomes, list) and len(outcomes) == 1):
            ok = False
            break
        rows.append({"patient": p, "outcome": str(outcomes[0])})
    if ok and len(rows) == len(PATIENTS):
        return pd.DataFrame(rows), str(PATIENT_INPUT_ROOT / "<patient>/retained_celltypes.json")

    import anndata as ad
    src = Path(CFG["input_h5ad"])
    assert src.is_file(), f"No usable patient-outcome mapping and {src} is absent"
    obs = ad.read_h5ad(src, backed="r").obs[[SAMPLE_COLUMN, OUTCOME_COLUMN]]
    series = derive_patient_outcomes(obs, SAMPLE_COLUMN, OUTCOME_COLUMN)
    out = series.rename("outcome").rename_axis("patient").reset_index()
    return out.astype(str), str(src)


OUTCOME_MAP, OUTCOME_SOURCE = load_patient_outcomes()
OUTCOME_MAP["patient"] = OUTCOME_MAP["patient"].astype(str)
OUTCOME_MAP["outcome"] = OUTCOME_MAP["outcome"].astype(str)

assert OUTCOME_MAP["patient"].is_unique, (
    f"A patient maps to more than one outcome in {OUTCOME_SOURCE}: "
    f"{OUTCOME_MAP[OUTCOME_MAP['patient'].duplicated(keep=False)].to_dict('records')}"
)
assert set(OUTCOME_MAP["outcome"]) <= set(OUTCOMES), (
    f"Unexpected outcome labels {sorted(set(OUTCOME_MAP['outcome']) - set(OUTCOMES))}"
)
missing_patients = sorted(set(PATIENTS) - set(OUTCOME_MAP["patient"]))
assert not missing_patients, f"No outcome for {missing_patients}"

OUTCOME_MAP = OUTCOME_MAP[OUTCOME_MAP["patient"].isin(PATIENTS)].sort_values("patient")
OUTCOME_COUNTS = OUTCOME_MAP["outcome"].value_counts().to_dict()
assert sum(OUTCOME_COUNTS.values()) == len(PATIENTS)
assert OUTCOME_COUNTS.get("Good") == 13 and OUTCOME_COUNTS.get("Poor") == 11, (
    f"Outcome split {OUTCOME_COUNTS}, plan expects 13 Good / 11 Poor"
)

tmp_out = OUT_PATIENT_OUTCOMES.with_suffix(".csv.tmp")
OUTCOME_MAP.to_csv(tmp_out, index=False)
publish_file(tmp_out, OUT_PATIENT_OUTCOMES)
OUTCOME_COUNTS

In [ ]:
# ============================================================
# Attach outcome to the secondary table and report its evaluable split.
# ============================================================
outcome_by_patient = dict(zip(OUTCOME_MAP["patient"], OUTCOME_MAP["outcome"]))
outcome_dtype = pd.CategoricalDtype(list(OUTCOMES), ordered=False)

SECONDARY["outcome"] = pd.Categorical(
    SECONDARY["patient"].astype(str).map(outcome_by_patient), dtype=outcome_dtype)
assert SECONDARY["outcome"].notna().all()

SECONDARY_SPLIT = (pd.Series({p: outcome_by_patient[p] for p in SECONDARY_PATIENTS})
                   .value_counts().to_dict())
assert SECONDARY_SPLIT == EXPECTED_SECONDARY_EVALUABLE_SPLIT, (
    f"Secondary-evaluable patients split {SECONDARY_SPLIT}, plan expects "
    f"{EXPECTED_SECONDARY_EVALUABLE_SPLIT}"
)

tmp_secondary = OUT_SECONDARY.with_suffix(".parquet.tmp")
pq.write_table(pa.Table.from_pandas(SECONDARY, preserve_index=False), tmp_secondary,
               compression="zstd")
publish_file(tmp_secondary, OUT_SECONDARY)

OUT_SECONDARY_SPEC.write_text(json.dumps({
    "definition": ("Mean CellPhoneDB interaction score over a fixed set of directional "
                   "core cell-type pairs, for one id_cp_interaction and one patient."),
    "core_celltype_rule": (f"retained (>= {CFG['patient_celltype_min_cells']} cells) in at "
                           f"least {SECONDARY_MIN_PATIENTS} of {len(PATIENTS)} patients"),
    "core_celltypes": CORE,
    "core_pairs": [f"{a}{CELLPAIR_SEPARATOR}{b}" for a, b in CORE_PAIRS],
    "n_core_pairs": len(CORE_PAIRS),
    "evaluability_rule": "all core pairs computed, otherwise the patient is not evaluable",
    "evaluable_patients": SECONDARY_PATIENTS,
    "n_evaluable_patients": len(SECONDARY_PATIENTS),
    "evaluable_split": SECONDARY_SPLIT,
    "non_evaluable_patients": sorted(set(PATIENTS) - set(SECONDARY_PATIENTS)),
}, indent=2))

print(f"secondary evaluable split: {SECONDARY_SPLIT}")
SECONDARY.head()

In [ ]:
# ============================================================
# Attach outcome to the long table LAST and republish it.
# ============================================================
SCORES["outcome"] = pd.Categorical(
    SCORES["patient"].astype(str).map(outcome_by_patient), dtype=outcome_dtype)
assert SCORES["outcome"].notna().all()
SCORES = SCORES.drop(columns=[c for c in ("is_nonzero",) if c in SCORES.columns])

tmp_scores = OUT_SCORES_LONG.with_suffix(".parquet.tmp")
pq.write_table(pa.Table.from_pandas(SCORES, preserve_index=False), tmp_scores, compression="zstd")
publish_file(tmp_scores, OUT_SCORES_LONG)
assert OUT_SCORES_LONG.is_file() and not tmp_scores.exists()

reread = pq.read_table(OUT_SCORES_LONG, columns=["patient", "id_cp_interaction", "sender",
                                                 "receiver", "score", "outcome"]).num_rows
assert reread == len(SCORES), f"Republished table has {reread} rows, expected {len(SCORES)}"
reread

## 13. Machine-readable success record

In [ ]:
# ============================================================
# Final success record for 03_.
# ============================================================
finished = datetime.now(timezone.utc).isoformat()
record = {
    "notebook": "03_extract_patient_endpoints.ipynb",
    "status": "success",
    "started_utc": started,
    "finished_utc": finished,
    "config_sha256": CFG_HASH,
    "utils_sha256": sha256_file(CODE_ROOT / "utils.py"),
    "utils_signatures": UTILS_SIGNATURES,
    "uses_pooled_information": False,
    "source_table": SOURCE_TABLE,
    "n_patients": len(PATIENTS),
    "patients": PATIENTS,
    "n_known_celltypes": len(KNOWN_CELLTYPES),
    "known_celltypes": KNOWN_CELLTYPES,
    "retained_celltypes_per_patient": {p: RETAINED[p] for p in PATIENTS},
    "n_union_interactions": N_INTERACTIONS,
    "n_union_directional_pairs": N_PAIRS,
    "n_endpoint_universe": N_UNIVERSE,
    "n_evaluated_rows": int(len(SCORES)),
    "n_nonzero_rows": int((SCORES["score"] != 0).sum()),
    "nonzero_fraction": float((SCORES["score"] != 0).mean()),
    "status_counts": {k: int(v) for k, v in AVAILABILITY_COUNTS.sum().items()},
    "unexpected_missing_total": UNEXPECTED_TOTAL,
    "unexpected_missing_explanation": "interaction_row_absent_for_patient",
    "unexpected_missing_unexplained": 0,
    "structural_missing_never_zero_filled": True,
    "outcome_source": OUTCOME_SOURCE,
    "outcome_counts": OUTCOME_COUNTS,
    "outcome_joined_after_qc": True,
    "secondary_core_celltypes": CORE,
    "n_secondary_core_pairs": len(CORE_PAIRS),
    "secondary_evaluable_patients": SECONDARY_PATIENTS,
    "secondary_evaluable_split": SECONDARY_SPLIT,
    "wide_matrix_written": bool(WIDE_WRITTEN),
    "outputs": {
        "scores_long": str(OUT_SCORES_LONG),
        "availability": str(OUT_AVAILABILITY),
        "endpoint_metadata": str(OUT_METADATA),
        "endpoint_qc": str(OUT_QC),
        "secondary": str(OUT_SECONDARY),
        "secondary_definition": str(OUT_SECONDARY_SPEC),
        "patient_outcomes": str(OUT_PATIENT_OUTCOMES),
        "wide": str(OUT_WIDE) if WIDE_WRITTEN else None,
    },
    "output_hashes": {
        name: sha256_file(Path(path))
        for name, path in {
            "scores_long": OUT_SCORES_LONG, "availability": OUT_AVAILABILITY,
            "endpoint_metadata": OUT_METADATA, "endpoint_qc": OUT_QC,
            "secondary": OUT_SECONDARY, "patient_outcomes": OUT_PATIENT_OUTCOMES,
        }.items()
    },
    "gate3": {"no_duplicate_endpoint_keys": True,
              "structural_missing_reconciled": True,
              "unexpected_missing_fully_explained": True},
}
SUCCESS_RECORD.write_text(json.dumps(record, indent=2, default=str))
assert SUCCESS_RECORD.is_file() and SUCCESS_RECORD.stat().st_size > 0
final_log = {"event": "notebook_complete", "notebook": record["notebook"],
             "status": record["status"], "n_evaluated_rows": record["n_evaluated_rows"],
             "unexpected_missing_total": record["unexpected_missing_total"],
             "timestamp_utc": finished}
log_jsonl(RUN_LOG, final_log)
print(f"GATE 3 PASSED — record at {SUCCESS_RECORD}")